# Transcript Data Processing
Combines every participant's transcript with corresponding labeled data into `raw_compiled_transcripts.csv`

# File paths

In [12]:
import pandas as pd
import os

# --- CẤU HÌNH ĐƯỜNG DẪN ---
BASE_DIR = os.getcwd() 
RAW_DATA_DIR = os.path.abspath(os.path.join(BASE_DIR, '..', 'data', 'raw_data'))
print(f"Đường dẫn dữ liệu: {RAW_DATA_DIR}")

# --- KIỂM TRA & LOAD FILE ---
try:
    # 1. Đọc dữ liệu từ file
    train_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'train_split_Depression_AVEC2017.csv'))
    dev_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'dev_split_Depression_AVEC2017.csv'))
    test_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'test_split_Depression_AVEC2017.csv'))
    
    # 2. Gộp lại
    Y_data = pd.concat([train_df, dev_df, test_df])
    
    # === [FIX QUAN TRỌNG] === 
    # Kiểm tra xem có bao nhiêu dòng bị lỗi (trống ID)
    nan_count = Y_data['Participant_ID'].isna().sum()
    if nan_count > 0:
        print(f"⚠️ Phát hiện {nan_count} dòng bị trống ID. Đang tự động loại bỏ...")
        Y_data = Y_data.dropna(subset=['Participant_ID']) # Xóa dòng trống
    
    # 3. Chuyển đổi sang số nguyên (Giờ thì an toàn rồi)
    Y_data['Participant_ID'] = Y_data['Participant_ID'].astype(int)
    Y_data.set_index('Participant_ID', inplace=True)
    
    print(f"✅ Đã tải thành công nhãn dữ liệu! Tổng số bệnh nhân hợp lệ: {len(Y_data)}")
    print("Ví dụ 3 dòng đầu tiên:")
    print(Y_data[['PHQ8_Binary', 'PHQ8_Score']].head(3))

    # Khai báo biến cho các ô sau
    transcript_directory = RAW_DATA_DIR 
    label_filepath = "dummy_variable_handled_above" 
    
except Exception as e:
    print("❌ LỖI CHI TIẾT:")
    print(e)

Đường dẫn dữ liệu: d:\TN-AI\automatic-depression-detector-main\automatic-depression-detector-main\data\raw_data
⚠️ Phát hiện 47 dòng bị trống ID. Đang tự động loại bỏ...
✅ Đã tải thành công nhãn dữ liệu! Tổng số bệnh nhân hợp lệ: 142
Ví dụ 3 dòng đầu tiên:
                PHQ8_Binary  PHQ8_Score
Participant_ID                         
303                     0.0         0.0
304                     0.0         6.0
305                     0.0         7.0


# Data Example

## X data

In [13]:
import pandas as pd 
import os

In [14]:
# --- SỬA ĐƯỜNG DẪN LOAD FILE MẪU ---
# Cấu trúc đúng là: data/raw_data/300_P/300_TRANSCRIPT.csv
p_id = 300
eg_transcript_filepath = os.path.join(transcript_directory, f"{p_id}_P", f"{p_id}_TRANSCRIPT.csv")

print(f"Đang đọc file mẫu tại: {eg_transcript_filepath}")

if os.path.exists(eg_transcript_filepath):
    # File này ngăn cách bằng dấu Tab (\t)
    df_transcript = pd.read_csv(eg_transcript_filepath, sep='\t')
    print("✅ Đọc thành công! Dữ liệu trông như sau:")
    display(df_transcript.head()) # Dùng display cho đẹp
else:
    print("❌ LỖI: Vẫn chưa tìm thấy file. Kiểm tra lại folder 300_P trong raw_data xem có file csv không?")

Đang đọc file mẫu tại: d:\TN-AI\automatic-depression-detector-main\automatic-depression-detector-main\data\raw_data\300_P\300_TRANSCRIPT.csv
✅ Đọc thành công! Dữ liệu trông như sau:


,start_time,stop_time,speaker,value
0,36.588,39.668,Ellie,hi i'm ellie thanks for coming in today
1,39.888,43.378,Ellie,i was created to talk to people in a safe and ...
2,43.728,48.498,Ellie,think of me as a friend i don't judge i can't ...
3,49.188,52.388,Ellie,i'm here to learn about people and would love ...
4,52.658,58.958,Ellie,i'll ask a few questions to get us started and...


## Y data

In [15]:
# --- SỬA LỖI LOAD NHÃN ---
# Thay vì đọc file, chúng ta dùng luôn dữ liệu đã tải ở bước 1
df_labels = Y_data.copy()

print("✅ Đã lấy nhãn thành công từ bộ nhớ!")
display(df_labels.head())

✅ Đã lấy nhãn thành công từ bộ nhớ!


,PHQ8_Binary,PHQ8_Score,Gender,PHQ8_NoInterest,PHQ8_Depressed,PHQ8_Sleep,PHQ8_Tired,PHQ8_Appetite,PHQ8_Failure,PHQ8_Concentrating,PHQ8_Moving,participant_ID
Participant_ID,,,,,,,,,,,,
303,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
304,0.0,6.0,0,0.0,1.0,1.0,2.0,2.0,0.0,0.0,0.0,NaN
305,0.0,7.0,1,0.0,1.0,1.0,2.0,2.0,1.0,0.0,0.0,NaN
310,0.0,4.0,1,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,NaN
312,0.0,2.0,1,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,NaN


# Process Data
1. Drop `start_time` and `stop_time` columns
2. Drop rows that contain prompts by Ellie
3. Concatenate the words spoken by the participant into a single chunk of text

In [16]:
def get_transcript(filepath):
    filename = os.path.basename(filepath)
    participant_no = int(filename[0:3])
    
    df_transcript = pd.read_csv(filepath, sep = '\t')  
    
    df_dropped_columns = df_transcript.drop(['start_time', 'stop_time'], axis = 1) # drop first 2 columns
    df_dropped_ellie = df_dropped_columns.set_index('speaker').drop(index = ['Ellie']) # drop anything Ellie says, she's not important
    df_cleaned = df_dropped_ellie.reset_index() 
    
    transcript_concat = '' 

    for index, row in df_cleaned.iterrows():
        if (index == 0):
            transcript_concat += str(row['value'])
        else:
            transcript_concat += ' ' + str(row['value'])
            
    return participant_no, transcript_concat

In [17]:
# Verify that we can get the right participant number and concatenated transcript

get_transcript(eg_transcript_filepath)

(300,
 "good atlanta georgia um my parents are from here um i love it i like the weather i like the opportunities um yes um it took a minute somewhat easy congestion that's it um i took up business and administration uh yeah i am here and there i'm on a break right now but i plan on going back in the uh next semester uh probably to open up my own business no um no specific reason i just don't travel a lot i'm pretty local once a year can you be a little bit more specific no answer i like reading books i enjoy i enjoy cooking um exercising is great i'm i'm i'm pretty good at it um yeah um probably about two weeks ago uh frustrated um i don't like bias um i don't like um when someone says they're gonna do something and they don't uh somewhat friendship i like to play sports i enjoy uh going out with friends and family playing games grandparents parents um yeah i mean they've always given me great advice they've always kept it real real close i would say going to college right after high 

# Compile all transcripts into a global dataframe

In [18]:
def combine_transcripts(directory, label_filepath):
    column_names_final = ["Participant_ID", "Transcript", "PHQ_Score", "PHQ_Binary"]
    df_final = pd.DataFrame(columns = column_names_final)
    
    csv_files = [pos_csv for pos_csv in os.listdir(directory) if pos_csv.endswith('.csv')]
    
    for filename in csv_files:
        filepath = directory + '/' + filename
        participant_no, transcript_concat = get_transcript(filepath)
        df_labels = pd.read_csv(label_filepath, index_col = "Participant_ID")
        PHQ_Binary = df_labels.loc[participant_no]["PHQ_Binary"]
        PHQ_Score = df_labels.loc[participant_no]["PHQ_Score"]

        new_row = {"Participant_ID": participant_no, "Transcript": transcript_concat, "PHQ_Score": PHQ_Score, "PHQ_Binary": PHQ_Binary}
        df_final = df_final.append(new_row, ignore_index = True)
    
    return df_final

In [20]:
# --- ĐOẠN CODE THAY THẾ HOÀN TOÀN HÀM combine_transcripts ---
import numpy as np

# Danh sách chứa kết quả
processed_data = []

print(f"🚀 Đang xử lý dữ liệu từ thư mục: {RAW_DATA_DIR}")

# Chúng ta sẽ duyệt qua danh sách ID có trong Y_data (biến nhãn đã tạo ở bước 1)
# Cách này an toàn tuyệt đối, không sợ đọc nhầm file rác
count = 0
for p_id in Y_data.index:
    # Tạo đường dẫn chính xác tới file transcript của bệnh nhân đó
    # Ví dụ: data/raw_data/303_P/303_TRANSCRIPT.csv
    file_path = os.path.join(RAW_DATA_DIR, f"{p_id}_P", f"{p_id}_TRANSCRIPT.csv")
    
    if os.path.exists(file_path):
        try:
            # Đọc file (cách nhau bằng dấu Tab)
            df = pd.read_csv(file_path, sep='\t')
            
            # Lọc lấy lời thoại của bệnh nhân (Participant)
            # Quan trọng: Phải chuyển về string để tránh lỗi
            participant_text = df[df['speaker'] == 'Participant']['value'].astype(str)
            
            # Nối lại thành 1 câu dài
            full_text = " ".join(participant_text)
            
            # Lưu vào danh sách
            processed_data.append([
                p_id, 
                full_text, 
                Y_data.loc[p_id, 'PHQ8_Binary'], 
                Y_data.loc[p_id, 'PHQ8_Score']
            ])
            count += 1
        except Exception as e:
            print(f"⚠️ Lỗi đọc file ID {p_id}: {e}")
    # Nếu không thấy file thì bỏ qua (không báo lỗi làm gì cho rối)

# Tạo bảng kết quả cuối cùng
df_final = pd.DataFrame(processed_data, columns=['Participant_ID', 'Text', 'PHQ8_Binary', 'PHQ8_Score'])

# Lưu ra file CSV
output_filename = "raw_compiled_transcripts.csv"
df_final.to_csv(output_filename, index=False)

print("-" * 30)
print(f"🎉 THÀNH CÔNG! Đã xử lý xong {count} bệnh nhân.")
print(f"📁 File kết quả đã lưu tại: {output_filename}")
print("-" * 30)
display(df_final.head())

🚀 Đang xử lý dữ liệu từ thư mục: d:\TN-AI\automatic-depression-detector-main\automatic-depression-detector-main\data\raw_data
------------------------------
🎉 THÀNH CÔNG! Đã xử lý xong 142 bệnh nhân.
📁 File kết quả đã lưu tại: raw_compiled_transcripts.csv
------------------------------


,Participant_ID,Text,PHQ8_Binary,PHQ8_Score
0,303,okay how 'bout yourself here in california yea...,0.0,0.0
1,304,i'm doing good um from los angeles california ...,0.0,6.0
2,305,i'm doing alright uh originally i'm from calif...,0.0,7.0
3,310,yes it's okay <laughter> fine <laughter> i liv...,0.0,4.0
4,312,yes fine how about you here yes the weather we...,0.0,2.0
